# C++ 第 5 章：模板与标准库算法

> **开始前：选择 C++17 内核。** 本章不能使用 C17 内核。在线实验使用固定数据或 `std::istringstream`，键盘输入 `std::cin` / `std::getline(std::cin, ...)` 放在本地完整程序中练习。

**先修**：C 部分的变量、控制语句、数组、指针和函数；按 C++ 章节顺序学习。

**本章目标**：用模板表达通用算法；理解迭代器、lambda、map 与 optional。

**学习方法**：先预测，再运行、修改并解释。块内局部变量可重复运行；修改函数、类或模板后，使用 Kernel → Restart Kernel and Run All Cells。各章互不依赖内核状态。

**保存作业**：见 [课程入口](https://github.com/luckymark/c_course#保存作业与继续学习)。

## 1. 模板保留类型信息

函数模板按使用的类型实例化。这里要求类型支持 < 和复制；并非任意类型都能使用。不同参数类型可能导致推导失败，例如 choose_larger(1, 2.5)，应先明确所需类型。返回值独立于局部对象寿命。

In [ ]:
template <typename T>
T choose_larger(T left, T right) { return left < right ? right : left; }

In [ ]:
#include <iostream>
#include <string>
{
    std::cout << choose_larger(3, 7) << ' ' << choose_larger(1.5, 2.5) << '\n';
    std::cout << choose_larger(std::string{"Alice"}, std::string{"Bob"}) << '\n';
}

## 2. 算法使用半开区间 [begin, end)

end 是尾后位置，不能解引用。std::sort 修改传入范围，先复制输入以保留原数据；count_if 只统计。lambda 是匿名函数：[] 描述捕获，参数与返回值描述计算。`[pass_line]` 按值捕获；`[&pass_line]` 引用捕获要求原对象存活。

排序比较必须满足严格弱序；用 > 降序，不要用 >=，相等元素与自身比较应为 false。

In [ ]:
#include <algorithm>
#include <vector>
#include <iostream>
{
    const std::vector<int> input{58, 90, 60, 90};
    auto ordered = input;
    std::sort(ordered.begin(), ordered.end(), [](int a, int b) { return a > b; });
    for (const int score : ordered) { std::cout << score << ' '; }
    int pass_line{60};
    std::cout << '\n' << std::count_if(input.begin(), input.end(), [pass_line](int score) { return score >= pass_line; }) << '\n';
}

## 3. map 与 optional

map 按键排序，适合姓名到分数的映射；unordered_map 不承诺按键顺序遍历。读取 map 用 find，避免 [] 意外插入缺失键。optional 表示“有值或无值”，不能用合法分数0表示未找到。解引用 optional 前检查是否有值。`const auto& [name, score]` 是 C++17 结构化绑定。

In [ ]:
#include <map>
#include <optional>
#include <string>
std::optional<int> lookup_score(const std::map<std::string, int>& scores, const std::string& name) {
    const auto found = scores.find(name);
    if (found == scores.end()) { return std::nullopt; }
    return found->second;
}

In [ ]:
#include <iostream>
{
    const std::map<std::string, int> scores{{"Bob", 0}, {"Alice", 85}};
    for (const auto& [name, score] : scores) { std::cout << name << ':' << score << '\n'; }
    const auto found = lookup_score(scores, "Bob");
    if (found) { std::cout << "found=" << *found << '\n'; }
    std::cout << "missing=" << !lookup_score(scores, "Carol").has_value() << '\n';
}

## 小练习：用 count_if 统计大于等于80的分数

只填写 BEGIN ANSWER 与 END ANSWER 之间的代码。先预测，再运行；初始 FAIL 表示尚未作答。完成后应全部 PASS。

In [ ]:
#include <algorithm>
#include <vector>
#include <iostream>
{
    const std::vector<std::vector<int>> cases{{}, {79, 80, 100}, {0}};
    const std::vector<int> expected{0, 2, 0};
    for (std::size_t i = 0; i < cases.size(); ++i) {
        int count = -1;
        const auto& scores = cases.at(i);
        (void)scores;
        /* BEGIN ANSWER */
        /* END ANSWER */
        std::cout << i << ": " << (count == expected.at(i) ? "PASS" : "FAIL") << '\n';
    }
}

<details><summary>提示</summary>

先检查空数据和边界，再处理一般情况；只计算题目需要的返回值，不改变输入。

</details>

<details><summary>参考答案：完成后再展开</summary>

```cpp
count = static_cast<int>(std::count_if(scores.begin(), scores.end(),
    [](int score) { return score >= 80; }));
```

</details>

## 本地实践：完整 C++17 程序

把代码保存为 `lesson.cpp`，在终端执行 `c++ -std=c++17 -Wall -Wextra -Wpedantic lesson.cpp -o lesson`，编译成功后运行 `./lesson`。Windows GCC 使用 `g++` 和 `-o lesson.exe`，PowerShell 运行 `.\lesson.exe`。完整程序不要直接放入在线单元或手动调用 main。

```cpp
#include <algorithm>
#include <iostream>
#include <vector>
int main() {
    int count{};
    if (!(std::cin >> count) || count < 0 || count > 100) { std::cout << "invalid\n"; return 1; }
    std::vector<int> scores;
    for (int i = 0; i < count; ++i) {
        int score{};
        if (!(std::cin >> score) || score < 0 || score > 100) { std::cout << "invalid\n"; return 1; }
        scores.push_back(score);
    }
    std::sort(scores.begin(), scores.end());
    for (const int score : scores) { std::cout << score << ' '; }
    std::cout << '\n';
}
```

验收输入与输出（`\n` 表示换行）：

- 输入 `"4 90 58 90 60"` → 输出 `"58 60 90 90 \n"`，退出码 0。
- 输入 `"0"` → 输出 `"\n"`，退出码 0。
- 输入 `"2 60"` → 输出 `"invalid\n"`，退出码 1。

## 分层练习与自查

1. **基础**：比较 sort 前后 input 与 ordered，解释为何输入未改变。
2. **改错**：为什么 find 返回 end 时不能读取 found->second？
3. **应用**：用 map 统计每个分数出现次数，再输出，测试重复分数与空输入。
4. **选做**：用 remove_if 与 erase 删除副本中的不及格成绩，解释算法返回的逻辑结尾与容器 size 的区别。